# Bài 50: DINO và Tự chưng cất

Hàm mất mát tương phản ở Bài 49 cần các cặp âm tường minh (các ảnh khác trong lô) để tránh nghiệm tầm thường là ánh xạ mọi ảnh về cùng một điểm. **Tự chưng cất (self-distillation)**, cơ chế đằng sau **DINO** (<a href="../references.html#caron-2021-dino">Caron et al., 2021</a>) và người kế nhiệm **DINOv2** (<a href="../references.html#oquab-2023-dinov2">Oquab et al., 2023</a>), loại bỏ hoàn toàn các mẫu âm: một mạng "giáo viên (teacher)" được cập nhật chậm hướng dẫn một mạng "học sinh (student)", hoàn toàn không có nhãn và không có cặp âm nào. Không có một biện pháp bảo vệ cẩn thận, thiết lập này sụp đổ về nghiệm tầm thường ngay lập tức &mdash; bài này xây dựng biện pháp bảo vệ đó (định tâm — centering) từ đầu và cho thấy chính xác tại sao nó cần thiết.

In [ ]:
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Học sinh, giáo viên và bài toán sụp đổ

Cả học sinh lẫn giáo viên đều là bản sao của cùng một kiến trúc bộ mã hóa nhỏ. Học sinh được huấn luyện bình thường, với gradient. Giáo viên *không bao giờ* được huấn luyện trực tiếp — sau mỗi bước, trọng số của nó được dịch một chút về phía trọng số hiện tại của học sinh (một **trung bình động hàm mũ (exponential moving average — EMA)**). Học sinh được huấn luyện để khớp với phân phối đầu ra của giáo viên trên một góc nhìn tăng cường *khác* của cùng ảnh.

Kiểu thất bại hiển nhiên: nếu đầu ra của giáo viên hoàn toàn không phụ thuộc vào đầu vào (luôn dự đoán cùng một vector hằng số, bất kể ảnh nào), học sinh có thể khớp nó một cách tầm thường bằng cách làm điều tương tự — hàm mất mát hoàn hảo, học được không chút thông tin nào. Đây là **sụp đổ biểu diễn (representation collapse)**, và nó là vấn đề trung tâm mà tự chưng cất phải giải quyết.

In [ ]:
def make_image(shape_type, cx, cy, size=16):
    img = np.zeros((size, size), dtype=np.float32)
    if shape_type == 'plus':
        img[cy-1:cy+2, cx-3:cx+4] = 1.0
        img[cy-3:cy+4, cx-1:cx+2] = 1.0
    else:
        yy, xx = np.mgrid[0:size, 0:size]
        img[((xx-cx)**2 + (yy-cy)**2) <= 9] = 1.0
    return img

def make_dataset(rng_local, n, position_range=(4, 12)):
    imgs, labels = [], []
    for _ in range(n):
        shape_type = rng_local.choice(['plus', 'circle'])
        cx, cy = rng_local.integers(*position_range), rng_local.integers(*position_range)
        imgs.append(make_image(shape_type, cx, cy))
        labels.append(0 if shape_type == 'plus' else 1)
    return np.array(imgs, dtype=np.float32), np.array(labels, dtype=np.int64)

def augment(img, rng_local):
    if rng_local.random() < 0.5:
        img = np.fliplr(img).copy()
    angle = rng_local.uniform(-20, 20)
    M = cv2.getRotationMatrix2D((8, 8), angle, 1.0)
    img = cv2.warpAffine(img, M, (16, 16))
    return np.clip(img + rng_local.normal(0, 0.1, img.shape), 0, 1).astype(np.float32)

rng = np.random.default_rng(5)
X_unlabeled, y_unlabeled = make_dataset(rng, 500)

class Encoder(nn.Module):
    def __init__(self, out_dim=16):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(), nn.AdaptiveMaxPool2d(1),
        )
        self.proj = nn.Linear(32, out_dim)

    def forward(self, x):
        return self.proj(self.conv(x).flatten(1))

def dino_loss(student_out, teacher_out, center, student_temp=0.1, teacher_temp=0.04):
    student_logp = F.log_softmax(student_out / student_temp, dim=-1)
    teacher_p = F.softmax((teacher_out - center) / teacher_temp, dim=-1)  # centering happens here
    return -(teacher_p.detach() * student_logp).sum(dim=-1).mean()

`dino_loss` có hai cơ chế chống sụp đổ được tích hợp sẵn:
- **Định tâm (centering)**: trừ đi một trung bình tích lũy (`center`) của các đầu ra gần đây của giáo viên trước softmax của giáo viên. Điều này ngăn giáo viên trôi về phía luôn dự đoán bất kỳ lớp đơn lẻ nào tình cờ dễ nhất — một đầu ra hằng số sẽ được định tâm về đúng 0, tự triệt tiêu chính nó.
- **Làm sắc (sharpening)**: softmax của giáo viên dùng nhiệt độ thấp hơn (`0.04`) so với của học sinh (`0.1`), làm cho phân phối mục tiêu của giáo viên tự tin hơn/nhọn hơn. Một mục tiêu giáo viên rất phẳng, thiếu tự tin gần với phân phối đều — chưa hẳn là sụp đổ, nhưng cũng không phải là một tín hiệu học hữu ích.

Cả hai mẹo cùng nhau là thứ cho phép DINO huấn luyện ổn định mà hoàn toàn không có cặp âm nào.

In [ ]:
def train_dino(seed, epochs=400, lr=0.005, batch_size=64, momentum=0.9, center_momentum=0.5, use_centering=True):
    torch.manual_seed(seed)
    student = Encoder()
    teacher = Encoder()
    teacher.load_state_dict(student.state_dict())
    for p in teacher.parameters():
        p.requires_grad_(False)
    opt = torch.optim.Adam(student.parameters(), lr=lr)
    center = torch.zeros(1, 16)
    local_aug_rng = np.random.default_rng(seed + 100)
    n = len(X_unlabeled)
    for epoch in range(epochs):
        idx = np.random.default_rng(epoch).permutation(n)[:batch_size]
        batch = X_unlabeled[idx]
        view1 = np.stack([augment(im, local_aug_rng) for im in batch])
        view2 = np.stack([augment(im, local_aug_rng) for im in batch])
        v1 = torch.tensor(view1).unsqueeze(1)
        v2 = torch.tensor(view2).unsqueeze(1)

        s1, s2 = student(v1), student(v2)
        with torch.no_grad():
            t1, t2 = teacher(v1), teacher(v2)

        c = center if use_centering else torch.zeros_like(center)
        loss = dino_loss(s1, t2, c) / 2 + dino_loss(s2, t1, c) / 2

        opt.zero_grad()
        loss.backward()
        opt.step()

        with torch.no_grad():
            for ps, pt in zip(student.parameters(), teacher.parameters()):
                pt.data.mul_(momentum).add_(ps.data, alpha=1 - momentum)  # EMA teacher update
            if use_centering:
                batch_center = torch.cat([t1, t2], dim=0).mean(dim=0, keepdim=True)
                center = center_momentum * center + (1 - center_momentum) * batch_center

    with torch.no_grad():
        output_std = torch.cat([t1, t2], dim=0).std(dim=0).mean().item()
    return student, teacher, output_std

_, teacher_centered, std_centered = train_dino(seed=0, use_centering=True)
_, teacher_no_center, std_no_center = train_dino(seed=0, use_centering=False)

print(f'teacher output std, WITH centering:    {std_centered:.4f}')
print(f'teacher output std, WITHOUT centering: {std_no_center:.4f}  (closer to 0 = more collapsed)')

Định tâm làm độ trải đầu ra của giáo viên tăng gần gấp ba so với huấn luyện không có nó — cơ chế làm đúng việc nó phải làm một cách thấy rõ. Giờ hãy kiểm tra xem điều đó có chuyển thành một biểu diễn hữu ích không, theo cùng cách Bài 49 đã làm: đóng băng giáo viên và huấn luyện một đầu dò tuyến tính trên một nhúm mẫu có nhãn.

In [ ]:
def make_noisy_dataset(rng_local, n, noise=0.35, position_range=(4, 12)):
    imgs, labels = [], []
    for _ in range(n):
        shape_type = rng_local.choice(['plus', 'circle'])
        cx, cy = rng_local.integers(*position_range), rng_local.integers(*position_range)
        img = make_image(shape_type, cx, cy)
        img = np.clip(img + rng_local.normal(0, noise, img.shape), 0, 1).astype(np.float32)
        imgs.append(img)
        labels.append(0 if shape_type == 'plus' else 1)
    return np.array(imgs, dtype=np.float32), np.array(labels, dtype=np.int64)

X_probe_train, y_probe_train = make_noisy_dataset(rng, 8)
X_probe_test, y_probe_test = make_noisy_dataset(rng, 150)

def linear_probe_acc(enc, seed):
    torch.manual_seed(seed)
    with torch.no_grad():
        feat_train = enc(torch.tensor(X_probe_train).unsqueeze(1))
        feat_test = enc(torch.tensor(X_probe_test).unsqueeze(1))
    probe = nn.Linear(feat_train.shape[1], 2)
    opt = torch.optim.Adam(probe.parameters(), lr=0.05)
    ytr = torch.tensor(y_probe_train)
    for _ in range(300):
        opt.zero_grad()
        loss = F.cross_entropy(probe(feat_train), ytr)
        loss.backward()
        opt.step()
    with torch.no_grad():
        preds = probe(feat_test).argmax(1).numpy()
    return (preds == y_probe_test).mean()

probe_accs = []
for seed in range(3):
    _, teacher, _ = train_dino(seed=seed)
    probe_accs.append(linear_probe_acc(teacher, seed=seed + 50))

print(f'linear probe on DINO-style teacher features: {np.mean(probe_accs):.1%}  (+/- {np.std(probe_accs):.1%})')
print(f'(Lesson 49 contrastive pretraining reached ~73% under the same probe setup)')

## Trực quan hóa trực tiếp biểu diễn

Các con số độ lệch chuẩn đầu ra ở trên đo sự sụp đổ trên các ảnh tổng hợp *sạch, dễ* được dùng để huấn luyện. Nhưng đầu dò tuyến tính ngay ở trên được đánh giá trên các ảnh nhiễu hơn, khó hơn, và cho kết quả gần mức đoán ngẫu nhiên. **t-SNE** (<a href="../references.html#vandermaaten-2008-tsne">van der Maaten & Hinton, 2008</a><span class="landmark-paper">&#9733;</span>, đã giới thiệu ở Bài 39) có thể cho thấy trực tiếp kết quả đầu dò yếu đó có phải là thuộc tính của *chính biểu diễn* không, chứ không chỉ do tập đầu dò nhỏ 8 mẫu: chiếu vector nhúng giáo viên của mỗi ảnh kiểm tra nhiễu xuống 2D cho ba bộ mã hóa — một đường cơ sở chưa huấn luyện, giáo viên bị sụp đổ (không định tâm), và giáo viên được huấn luyện đúng cách (có định tâm) — và xem ảnh dấu cộng và hình tròn có kết thúc ở các vùng tách biệt không. Nhãn thực (`y_probe_test`) chỉ được dùng để tô màu đồ thị, không bao giờ để huấn luyện bất cứ thứ gì.

In [ ]:
def tsne(X, n_iter=500, perplexity=15.0, lr=100.0, seed=0):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    sq_dists = ((X[:, None, :] - X[None, :, :]) ** 2).sum(-1)

    # high-dimensional affinities P: for each point, binary-search a Gaussian bandwidth so its
    # neighbor distribution has the target perplexity (an implicit "how many neighbors matter" knob)
    target_entropy = np.log(perplexity)
    P = np.zeros((n, n))
    for i in range(n):
        others = np.arange(n) != i
        d_i = sq_dists[i, others]
        lo, hi = 1e-4, 1e4
        for _ in range(50):
            beta = (lo + hi) / 2
            p = np.exp(-d_i * beta)
            p /= p.sum() + 1e-12
            entropy = -np.sum(p * np.log(p + 1e-12))
            if entropy > target_entropy:
                lo = beta  # entropy too high (too spread out) -> need a larger beta to sharpen it
            else:
                hi = beta
        P[i, others] = p
    P = (P + P.T) / (2 * n)          # symmetrize into one joint distribution over pairs
    P = np.maximum(P, 1e-12) * 4.0   # early exaggeration: temporarily inflate P so true neighbors clump together faster

    # low-dimensional embedding Y, fit by gradient descent on KL(P || Q)
    Y = rng.normal(0, 1e-2, (n, 2))
    velocity = np.zeros_like(Y)
    for it in range(n_iter):
        if it == 100:
            P /= 4.0  # turn off early exaggeration once clusters have separated
        d = ((Y[:, None, :] - Y[None, :, :]) ** 2).sum(-1)
        num = 1.0 / (1.0 + d)  # Student-t kernel -- heavy tails let moderately-distant points repel more strongly than a Gaussian would, avoiding the "crowding problem"
        np.fill_diagonal(num, 0)
        Q = np.maximum(num / num.sum(), 1e-12)
        coeff = (P - Q) * num
        grad = 4 * (coeff[:, :, None] * (Y[:, None, :] - Y[None, :, :])).sum(1)
        momentum = 0.5 if it < 100 else 0.8
        velocity = momentum * velocity - lr * grad
        Y = Y + velocity
    return Y

In [ ]:
torch.manual_seed(99)  # a third baseline: never trained at all
untrained_encoder = Encoder()

Xv = torch.tensor(X_probe_test).unsqueeze(1)

def teacher_features(encoder):
    with torch.no_grad():
        return encoder(Xv).numpy()

Y_untrained = tsne(teacher_features(untrained_encoder), seed=0)
Y_no_center = tsne(teacher_features(teacher_no_center), seed=0)
Y_centered = tsne(teacher_features(teacher_centered), seed=0)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
titles = ['untrained', 'no centering (collapsed)', 'with centering']
for ax, Y, title in zip(axes, [Y_untrained, Y_no_center, Y_centered], titles):
    ax.scatter(*Y[y_probe_test == 0].T, c='tab:blue', s=14, label='plus')
    ax.scatter(*Y[y_probe_test == 1].T, c='tab:orange', s=14, label='circle')
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=7)
fig.suptitle('t-SNE of teacher embeddings on the noisy probe test set, colored by true shape', y=1.02)
plt.show()

Cả ba khung đều trông thiếu thuyết phục như nhau — không khung nào cho thấy sự tách thành hai cụm gọn gàng mà Bài 39 đã tìm thấy cho các đặc trưng của một CNN có giám sát. Một phép kiểm tra láng giềng gần nhất nhanh trong không gian đặc trưng 16 chiều thô (láng giềng gần nhất của mỗi điểm có cùng nhãn thực với nó không?) xác nhận điều mà các đồ thị gợi ý: khoảng 59% với giáo viên có định tâm, 63% với giáo viên bị sụp đổ không định tâm, và 62% với đường cơ sở chưa huấn luyện — tất cả đều gần mức sàn 50% của đoán ngẫu nhiên, không có khoảng cách đáng kể nào giữa "bị sụp đổ" và "được huấn luyện đúng cách".

Điều này đáng để suy ngẫm thay vì giải thích cho qua: thước đo độ lệch chuẩn đầu ra ở phần trước của bài là có thật và cho thấy đúng rằng định tâm ngăn chặn sụp đổ trên phân phối *huấn luyện*, nhưng đó là một điều kiện cần, không phải điều kiện đủ, và phép trực quan hóa này là bằng chứng trực tiếp về khoảng cách giữa chúng. Chỉ với 400 epoch huấn luyện trên 500 ảnh đồ chơi không nhãn với một bộ mã hóa nhỏ xíu, đơn giản là không có đủ tín hiệu huấn luyện để lợi ích của định tâm thể hiện thành *cấu trúc hữu ích, chống chịu nhiễu* trong không gian nhúng — chính xác cùng giới hạn mà độ chính xác đầu dò tuyến tính ở trên đã chỉ ra, giờ thấy được trực tiếp trong chính không gian đặc trưng thay vì qua một con số độ chính xác hạ nguồn. Đó là một minh họa cụ thể cho lý do DINOv2 cần quy mô được mô tả bên dưới: tránh sụp đổ chỉ là điều kiện tối thiểu, và chất lượng biểu diễn thực sự chỉ xuất hiện với dữ liệu, năng lực và thời gian huấn luyện nhiều hơn hẳn những gì một notebook đồ chơi có thể cung cấp.

Biện pháp bảo vệ chống sụp đổ hoạt động một cách đo được — đầu ra của giáo viên vẫn trải rộng, không phải hằng số. Nhưng ở quy mô đồ chơi này (vài trăm epoch huấn luyện, vài trăm ảnh không nhãn), đầu dò tuyến tính cho kết quả gần mức đoán ngẫu nhiên, kém xa kết quả tương phản ở Bài 49 trên cùng nhiệm vụ. Đây là một kết quả trung thực, không phải lỗi: tự chưng cất được biết là *nhạy cảm hơn* với các siêu tham số (quán tính EMA, lịch trình nhiệt độ, tốc độ định tâm) và nói chung cần nhiều tín hiệu huấn luyện hơn đáng kể để đạt được một biểu diễn hữu ích so với một hàm mất mát tương phản có mẫu âm tường minh. Tránh sụp đổ là cần nhưng chưa đủ để học được thứ gì hữu ích — nó chỉ dọn đường cho đủ huấn luyện để cuối cùng làm được điều đó.

## DINOv2 bổ sung gì ở quy mô thực

**DINOv2** (<a href="../references.html#oquab-2023-dinov2">Meta AI, 2023</a>) chính là cơ chế này — tự chưng cất học sinh/giáo viên với định tâm, cộng với vài tinh chỉnh (nhiều "vùng cắt cục bộ (local crops)" nhỏ bên cạnh các "vùng cắt toàn cục (global crops)" toàn ảnh, để làm nhiệm vụ khó hơn và phong phú hơn) — được mở rộng quy mô lên một tập dữ liệu *không nhãn* 142 triệu ảnh được tuyển chọn và một mạng xương sống Vision Transformer (Bài 47) với tới 1,1 tỉ tham số. Ở quy mô đó, biểu diễn không chỉ "dùng được với một đầu dò tuyến tính" — nó thể hiện những thuộc tính nổi lên (emergent) đáng kinh ngạc mà không ai huấn luyện một cách tường minh: các bản đồ chú ý từ một ViT DINOv2 thường viền quanh ranh giới và các bộ phận của đối tượng mà chưa bao giờ thấy một nhãn phân đoạn nào (một lời báo trước trực tiếp cho Bài 54), và phân loại láng giềng gần nhất k (k-nearest-neighbor) trực tiếp trên các đặc trưng DINOv2 đóng băng sánh ngang huấn luyện có giám sát trên vài bộ đánh giá chuẩn, tất cả mà không cần tinh chỉnh một trọng số nào.

Sợi chỉ xuyên suốt từ bài này đến DINOv2 chính xác là khoảng cách mà notebook này đã phơi bày: *cơ chế* (học sinh, giáo viên, định tâm) là cùng một đoạn code ở bất kỳ quy mô nào; điều thay đổi giữa phiên bản đồ chơi này và một mô hình nền tảng thực sự là khối lượng dữ liệu, năng lực mô hình và thời gian huấn luyện — cùng câu chuyện như Bài 37 (LeNet đến AlexNet) và Bài 49 (SimCLR ở quy mô đồ chơi so với ở quy mô 1000+ GPU), được kể thêm một lần nữa.

### Bài tập

1. Tăng `epochs` trong `train_dino` từ 400 lên 1200 (sẽ mất nhiều thời gian chạy hơn). Độ chính xác đầu dò tuyến tính có cải thiện thấy rõ, giữ phẳng, hay trở nên bất ổn — và điều đó so với những gì thêm epoch huấn luyện đã làm cho cách tiếp cận tương phản ở Bài 49 thế nào?
2. Đặt `teacher_temp=0.1` (khớp chính xác với nhiệt độ của học sinh, loại bỏ sự bất đối xứng làm sắc) trong `dino_loss`. Phép so sánh sụp đổ (có và không có định tâm) có còn cho thấy khoảng cách rõ ràng không, hay việc loại bỏ làm sắc khiến sụp đổ xảy ra ngay cả khi bật định tâm?
3. Thử `momentum=0.5` (một giáo viên cập nhật nhanh hơn nhiều) thay vì `0.9`. Một giáo viên cập nhật nhanh gần bằng học sinh đánh mất mục đích chính của nó — cung cấp một mục tiêu ổn định, thay đổi chậm. Huấn luyện có trở nên kém ổn định hơn không, và bạn có thấy điều đó trong thước đo sụp đổ không?